# Monitoring with MLflow

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pecca-core/pecca/blob/main/examples/notebooks/monitoring_mlflow.ipynb)

Pecca stores each model version in your registry. With `workspace.registry: mlflow://...` every version becomes an MLflow run (params, metrics, artefacts) plus a `pyfunc` model, and call state and decision logs live next to them. Here a local MLflow server plays the role of your tracking server.

In [1]:
%pip install -q pecca requests mlflow

In [2]:
import json, os, re, shutil, subprocess, tempfile, time
from pathlib import Path

import pandas as pd
import pecca
from pecca.core import context
from pecca.data.dataset import Dataset, canonicalize
from pecca.data.synthetic import generate

HERE = Path.cwd()                                   # the notebook's folder (examples/notebooks in the repo)
work = tempfile.mkdtemp()
os.chdir(work)
os.environ["PECCA_HOME"] = os.path.join(work, ".pecca")
os.environ["TQDM_DISABLE"] = "1"
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "1"
import logging
try:
    import mlflow  # MLflow resets its logger level on import, so quiet it afterwards
    logging.getLogger("mlflow").setLevel(logging.ERROR)
except ImportError:
    pass

# Docker-backed cells run only when Docker is reachable and PECCA_SKIP_DOCKER != 1 (CI sets it).
USE_DOCKER = (os.environ.get("PECCA_SKIP_DOCKER") != "1" and shutil.which("docker") is not None
              and subprocess.run(["docker", "info"], capture_output=True).returncode == 0)
print("docker:", "available" if USE_DOCKER else "skipped (PECCA_SKIP_DOCKER=1 or Docker not reachable)")

def sh(*args, check=True):
    return subprocess.run([str(a) for a in args], capture_output=True, text=True, check=check)

import atexit
def cleanup_later(*cmd):
    """Run `cmd` when the kernel exits, so containers are removed even if a cell fails."""
    atexit.register(lambda: subprocess.run([str(c) for c in cmd], capture_output=True))

def free_port():
    import socket
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

def wait_for(check, what, seconds=90):
    end = time.time() + seconds
    while time.time() < end:
        try:
            if check():
                return
        except Exception:
            pass
        time.sleep(1)
    raise TimeoutError(f"{what} did not become ready in {seconds}s")

def train_demo(call="route_rfi", rows=3000, candidates=("tfidf_linear",), **kw):
    """Train a small model on the synthetic demo data (LLM answers + human corrections)."""
    df = generate(rows)
    df["text"] = df.email_subject + "\n\n" + df.email_body
    cols = {"input": "text", "llm_output": "llm_rfi", "human_label": "human_rfi", "call_name": {"literal": call.split("/")[-1]}}
    return pecca.train(call, Dataset(canonicalize(df, cols), cols), candidates=list(candidates), **kw)

docker: available


## 1. Start a local MLflow server

In [3]:
import socket, sys
port = free_port()
mlflow_proc = subprocess.Popen([sys.executable, "-m", "mlflow", "server", "--host", "127.0.0.1", "--port", str(port),
                                "--backend-store-uri", f"sqlite:///{work}/mlflow.db", "--default-artifact-root", f"{work}/artifacts"],
                               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
import requests
wait_for(lambda: requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200, "MLflow server")
atexit.register(mlflow_proc.terminate)
print("MLflow server is up (UI on http://127.0.0.1:<port>)")

MLflow server is up (UI on http://127.0.0.1:<port>)


## 2. Use it as Pecca's registry

In [4]:
Path("pecca.yaml").write_text(f"version: 1\nworkspace:\n  name: default\n  registry: mlflow://http/127.0.0.1:{port}\nprojects:\n  default: {{governance: {{extends: none}}}}\n")
context.reset()
v1 = train_demo(rows=1500)
v2 = train_demo(rows=3000)
print("versions:", v1.version, v2.version)

versions: v1 v2


In [5]:
# The "LLM": replay the simulated LLM's answers for a few new emails (no API needed).
new = generate(30, seed=11)
texts = (new.email_subject + "\n\n" + new.email_body).tolist()
said = dict(zip(texts, new.llm_rfi))

@pecca.replace("route_rfi", mode="shadow")
def route_rfi(text: str) -> str:
    return said[text]                      # stand-in for my_llm.classify(text), unchanged

pecca.promote("route_rfi", "shadow")
for t in texts:
    route_rfi(t)
from pecca.runtime.logging_sink import flush_all
flush_all()
print(f"logged {len(texts)} shadow calls")

logged 30 shadow calls


## 3. What MLflow now shows

In [6]:
import mlflow
client = mlflow.MlflowClient(f"http://127.0.0.1:{port}")
exp = client.get_experiment_by_name("pecca/default/default/route_rfi")
runs = client.search_runs([exp.experiment_id], order_by=["attributes.start_time ASC"])
table = pd.DataFrame([{"run": r.data.tags.get("pecca.kind"), "version": r.data.tags.get("pecca.version", "-"),
                       "candidate": r.data.params.get("candidate", "-"), "macro_f1": r.data.metrics.get("metric"),
                       "threshold": r.data.metrics.get("threshold"), "artefacts": len(client.list_artifacts(r.info.run_id))} for r in runs])
print(f"experiment: {exp.name}")
table

experiment: pecca/default/default/route_rfi


,run,version,candidate,macro_f1,threshold,artefacts
0,version,v1,tfidf_linear,0.897894,0.991749,2
1,state,-,-,NaN,NaN,2
2,version,v2,tfidf_linear,0.917297,0.593069,2


In [7]:
version_run = [r for r in runs if r.data.tags.get("pecca.version") == "v2"][0]
print("v2 artefacts:", sorted(a.path for a in client.list_artifacts(version_run.info.run_id)))
state_run = [r for r in runs if r.data.tags.get("pecca.kind") == "state"][0]
print("state run artefacts:", sorted(a.path for a in client.list_artifacts(state_run.info.run_id)))
print("pyfunc model:", "ok" if not version_run.data.tags.get("pecca.pyfunc_error") else version_run.data.tags["pecca.pyfunc_error"])
state = context.get_call("route_rfi").state()
print("current state:", {"mode": state.mode, "current_version": state.current_version})

v2 artefacts: ['metadata.json', 'model_artefacts']
state run artefacts: ['logs', 'state.json']
pyfunc model: ok
current state: {'mode': 'shadow', 'current_version': 'v2'}


## 4. Clean up

In [8]:
mlflow_proc.terminate(); mlflow_proc.wait(10)
print("MLflow server stopped")

MLflow server stopped
